In [43]:
import pandas as pd
from DEplots.volcano import volcano_from_deseq_result
import plotly.graph_objs as go
import numpy as np


In [44]:
f1 = "../Pipeline/RUNS/LightRunPlusPuromycin/PipelineData/DESeqResults/DESeqResult_cM_vs_bC.tsv"
f2 = "../Pipeline/RUNS/LightRunMinusPuromycin/PipelineData/DESeqResults/DESeqResult_cM_vs_bC.tsv"
add_data = "../initialData/locusTagtoGeneName.csv"

df1 = pd.read_csv(f1, sep="\t")
df2 = pd.read_csv(f2, sep="\t")


add_data = pd.read_csv(add_data, sep=",")
df1 = df1.merge(add_data, left_index=True, right_on="locus_tag")
df1 = df1.set_index("locus_tag")

highlight = {}
config = {
        "membraneColor": "#2ca02c",
        "cytosolColor": "#002695"
}

up_in_m_p = df2.loc[(df2["log2FoldChange"] >= 0.8) & (df2["padj"] <= 0.05)].index

down_in_m_p = df2.loc[(df2["log2FoldChange"] <= -0.8) & (df2["padj"] <= 0.05)].index

highlight["Membrane in -P"] = (config["membraneColor"], up_in_m_p)
highlight["Cytosol in -P"] = (config["cytosolColor"], down_in_m_p)

In [45]:
fig = volcano_from_deseq_result(
        df1,
        name_col="gene_name",
        highlight=highlight,
        lfc_cutoff=0.8,
        padj_cutoff=0.05,
        condition_name="Membrane",
        base_name="Cytosol",
        highlight_up_color=config["membraneColor"],
        highlight_down_color=config["cytosolColor"],
        opacity=0.3
    )
fig.update_layout(template="plotly_white")

/home/rabsch/miniconda/envs/RlocSeq/lib/python3.11/site-packages/DEplots/volcano.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [46]:
fig.write_image("plusPcomparison.svg")
fig.write_html("plusPcomparison.html")

In [47]:

df = "../Pipeline/Summary/joinedDESeq_cM_vs_bC.tsv"
df = pd.read_csv(df, sep="\t")

df = df.rename({
        "log2FoldChange_minusPuromycin": "log2FoldChange",
        "padj_minusPuromycin": "padj"
    }, axis=1)

config["RNA_TYPE_COLORS"]  = {
    "5UTR": "#8f6b30",        # Orange Red
  "5'UTR": "#8f6b30",        # Orange Red
  "3UTR": "rgb(160, 82, 45)",       # Sienna
  "3'UTR": "rgb(160, 82, 45)",       # Sienna
  "asRNA": "#ffe863",      # Deep Sky Blue
  "ncRNA": "rgb(0, 128, 0)",        # Green
  "tRNA": "rgb(255, 105, 180)",     # Hot Pink
  "CRISPR": "rgb(255, 215, 0)",     # Gold
  "mRNA": "#f5c2ed",         # Blue
  "CDS": "#f5c2ed",         # Blue
  "gene": "rgb(219, 219, 219)",         # Blue
  "default": "#7CB9E8",         # Blue

}
gsea_file = "../Pipeline/rnaTypeGSEA/gseaEnrich_purominusPuromycin.tsv"

gsea = pd.read_csv(gsea_file, sep="\t")
highlight = {key: (color, df[df[key] == True].index) for key, color in config["RNA_TYPE_COLORS"].items() if key in gsea.ID.tolist()}

fig = volcano_from_deseq_result(
        df,
        name_col="gene_name",
        highlight=highlight,
        lfc_cutoff=0.8,
        padj_cutoff=0.05,
        condition_name="Membrane",
        base_name="Cytosol",
        highlight_up_color=config["membraneColor"],
        highlight_down_color=config["cytosolColor"],
        opacity=0.3
    )
annotations = {
    "cupA": {"x": 1.55, "y": 6.2},
    "psaA": {"x": 0.15, "y": 4},
    "rpl11": {"x": 0.55, "y": 6},
    "rnpB": {"x": -1.5, "y": 9},
    "fusB": {"x": -1, "y": 6},
    "rpl10": {"x": -0.2, "y": 1.75},
    "psbA2": {"x": 1.1, "y": 8},
    "rpl12": {"x": 2, "y": 10},
    "rimO": {"x": None, "y": None}
}
for anno, data in annotations.items():
    sdf = df[df["gene_name"] == anno]
    print(data)
    for _, n in sdf.iterrows():
        fig.add_annotation(
            x=n["log2FoldChange"],
            y=-1 * np.log10(n["padj"]),
            ax=data["x"],
            ay=data["y"],
            text=anno,
            showarrow=True,
            axref="x",
            ayref="y",
            arrowwidth=1,
            arrowsize=0.5,
            arrowcolor="black",
            #font=dict(size=lout["annotationsize"])

        )
fig.update_layout(template="plotly_white")


/home/rabsch/miniconda/envs/RlocSeq/lib/python3.11/site-packages/DEplots/volcano.py:23: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



{'x': 1.55, 'y': 6.2}
{'x': 0.15, 'y': 4}
{'x': 0.55, 'y': 6}
{'x': -1.5, 'y': 9}
{'x': -1, 'y': 6}
{'x': -0.2, 'y': 1.75}
{'x': 1.1, 'y': 8}
{'x': 2, 'y': 10}
{'x': None, 'y': None}


In [48]:
fig.write_image("minusPcomparison.svg")
fig.write_html("minusPcomparison.html")